# TP8 — Le TP7 sur Databricks Free Edition
**Module Big Data — ISI, DITI 4 — 2026-2027**

Ce notebook accompagne l'énoncé du TP8 (plateforme des TP). Exécutez les cellules **dans l'ordre**, une par une (`Maj + Entrée`).

* Avant la première cellule : en haut à droite, **Connect** → **Serverless**.
* Les cellules marquées `# À FAIRE` sont à compléter : vous y reprenez le code que vous avez écrit au TP7.
* Les cellules marquées `# ERREUR ATTENDUE` doivent échouer : notez le message d'erreur.

## 1. Paramètres
Le catalogue par défaut d'un espace Free Edition s'appelle normalement `workspace`. Vérifiez-le dans **Catalog** (barre latérale) ; s'il porte un autre nom, modifiez `CATALOGUE` ici **et** dans les cellules `%sql`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import time

CATALOGUE = "workspace"
SCHEMA = "tp8"
VOLUME = "donnees"
BASE = f"/Volumes/{CATALOGUE}/{SCHEMA}/{VOLUME}"

def chrono(action):
    t = time.time()
    action()
    return round(time.time() - t, 1)

print(spark.version, BASE)

## 2. Le stockage : schéma, volume et fichiers
Un **schéma** regroupe des tables ; un **volume** est un espace de fichiers géré par Unity Catalog. Ensemble, ils remplacent le HDFS des TP précédents.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGUE}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGUE}.{SCHEMA}.{VOLUME}")
display(spark.sql(f"SHOW VOLUMES IN {CATALOGUE}.{SCHEMA}"))

Le petit fichier `purchases.txt` des TP2, TP5 et TP7 (19 ventes) est écrit directement dans le volume, comme un fichier ordinaire :

In [ ]:
lignes = [
    ("2012-12-31", "17:59", "Chula Vista", "Music", "380.67", "Visa"),
    ("2012-12-31", "17:59", "Hialeah", "Toys", "115.21", "MasterCard"),
    ("2012-12-31", "17:59", "Indianapolis", "Men's Clothing", "158.28", "MasterCard"),
    ("2012-12-31", "17:59", "Norfolk", "Garden", "414.09", "MasterCard"),
    ("2012-12-31", "17:59", "Baltimore", "DVDs", "467.3", "Visa"),
    ("2012-12-31", "17:59", "Santa Ana", "Video Games", "144.73", "Visa"),
    ("2012-12-31", "17:59", "Gilbert", "Consumer Electronics", "354.66", "Discover"),
    ("2012-12-31", "17:59", "Memphis", "Sporting Goods", "124.79", "Amex"),
    ("2012-12-31", "17:59", "Chicago", "Men's Clothing", "386.54", "MasterCard"),
    ("2012-12-31", "17:59", "Birmingham", "CDs", "118.04", "Cash"),
    ("2012-12-31", "17:59", "Las Vegas", "Health and Beauty", "420.46", "Amex"),
    ("2012-12-31", "17:59", "Wichita", "Toys", "383.9", "Cash"),
    ("2012-12-31", "17:59", "Tucson", "Pet Supplies", "268.39", "MasterCard"),
    ("2012-12-31", "17:59", "Glendale", "Women's Clothing", "68.05", "Amex"),
    ("2012-12-31", "17:59", "Albuquerque", "Toys", "345.7", "MasterCard"),
    ("2012-12-31", "17:59", "Rochester", "DVDs", "399.57", "Amex"),
    ("2012-12-31", "17:59", "Greensboro", "Baby", "277.27", "Discover"),
    ("2012-12-31", "17:59", "Arlington", "Women's Clothing", "134.95", "MasterCard"),
    ("2012-12-31", "17:59", "Corpus Christi", "DVDs", "441.61", "Discover"),
]
with open(f"{BASE}/purchases.txt", "w") as f:
    for l in lignes:
        f.write("\t".join(l) + "\n")
print(open(f"{BASE}/purchases.txt").read()[:200])

**Téléversement de `regions.csv`** (le référentiel du TP7, section 6) : suivez la section 3.3 de l'énoncé (**New → Add or upload data → Upload files to a volume**), puis vérifiez :

In [ ]:
display(dbutils.fs.ls(BASE))

## 3. Générer 3 millions de ventes… avec Spark
Au TP5, `gen_ventes.py` tournait sur la VM, dans un seul processus Python. Ici, on n'a pas de VM : on génère les données **avec Spark**, en parallèle.

In [ ]:
villes = ["Chula Vista", "Hialeah", "Indianapolis", "Norfolk", "Baltimore", "Santa Ana",
          "Gilbert", "Memphis", "Chicago", "Birmingham", "Las Vegas", "Wichita", "Tucson",
          "Glendale", "Albuquerque", "Rochester", "Greensboro", "Arlington", "Corpus Christi"]
categories = ["Music", "Toys", "Men's Clothing", "Garden", "DVDs", "Video Games",
              "Consumer Electronics", "Sporting Goods", "CDs", "Health and Beauty",
              "Pet Supplies", "Women's Clothing", "Baby", "Books", "Computers"]
paiements = ["Visa", "MasterCard", "Amex", "Discover", "Cash"]

def au_hasard(liste, graine):
    """Une valeur de la liste, tirée au hasard pour chaque ligne."""
    choix = F.array(*[F.lit(v) for v in liste])
    return F.element_at(choix, (F.floor(F.rand(graine) * len(liste)) + 1).cast("int"))

def entier(mini, maxi, graine):
    return (F.floor(F.rand(graine) * (maxi - mini + 1)) + mini).cast("int")

gen = (spark.range(1, 3_000_001).withColumnRenamed("id", "id_vente")
       .select("id_vente",
               F.format_string("2012-%02d-%02d", entier(1, 12, 1), entier(1, 28, 2)).alias("date"),
               F.format_string("%02d:%02d", entier(9, 20, 3), entier(0, 59, 4)).alias("heure"),
               au_hasard(villes, 5).alias("ville"),
               au_hasard(categories, 6).alias("categorie"),
               F.round(F.rand(7) * 495 + 5, 2).alias("montant"),
               au_hasard(paiements, 8).alias("paiement")))
display(gen.limit(5))

In [ ]:
# Même format que purchases_big.txt du TP5 : 6 colonnes séparées par des tabulations, sans identifiant
(gen.drop("id_vente")
    .write.mode("overwrite")
    .csv(f"{BASE}/purchases_big", sep="\t"))
display(dbutils.fs.ls(f"{BASE}/purchases_big"))

## 4. Rejouer le TP7
### 4.1 Lire avec un schéma

In [ ]:
schema = StructType([
    StructField("date", StringType()),
    StructField("heure", StringType()),
    StructField("ville", StringType()),
    StructField("categorie", StringType()),
    StructField("montant", DoubleType()),
    StructField("paiement", StringType()),
])
ventes = spark.read.csv(f"{BASE}/purchases_big", sep="\t", schema=schema)
petit = spark.read.csv(f"{BASE}/purchases.txt", sep="\t", schema=schema)
ventes.printSchema()
ventes.count()

### 4.2 Le CA par ville, en DataFrame et en SQL

In [ ]:
ca_df = (ventes.groupBy("ville")
               .agg(F.sum("montant").alias("ca"))
               .orderBy(F.desc("ca")))
display(ca_df)

In [ ]:
ventes.createOrReplaceTempView("ventes")
petit.createOrReplaceTempView("petit")

In [ ]:
%sql
SELECT ville, ROUND(SUM(montant), 2) AS ca
FROM ventes
GROUP BY ville
ORDER BY ca DESC

In [ ]:
chrono(lambda: ca_df.collect()), chrono(lambda: ca_df.collect())

In [ ]:
ca_df.explain("formatted")

Sous la cellule qui a exécuté `ca_df.collect()`, ouvrez le **profil de la requête** (section 4.3 de l'énoncé).

### 4.3 Jointure et fenêtres (TP7, sections 6 et 7)

In [ ]:
# À FAIRE — lisez regions.csv depuis le volume (avec en-tête), puis calculez le CA par région, trié (TP7, section 6).


In [ ]:
# À FAIRE — les deux meilleures villes de chaque région, avec une fonction de fenêtre (TP7, section 7).


### 4.4 Les analyses du TP7 (section 8)
Collez vos versions DataFrame **ou** SQL (`%sql`) des analyses 1 à 8 du TP7, une par cellule. Vérifiez-les sur la vue `petit`.

In [ ]:
# À FAIRE — analyse 2 — CA par moyen de paiement et part en %.


In [ ]:
# À FAIRE — analyse 4 — panier moyen par ville.


In [ ]:
# À FAIRE — analyse 6 — top 3 des catégories dans chaque région.


In [ ]:
# À FAIRE — analyse 7 — part de chaque ville dans le CA de sa région (en %).


In [ ]:
# À FAIRE — analyse 8 — CA mensuel et CA cumulé depuis janvier.


## 5. Ce qui ne passe plus
Chacune des trois cellules suivantes fonctionnait au TP5 ou au TP7. Exécutez-les et **notez le message d'erreur** (le texte exact peut varier).

In [ ]:
# ERREUR ATTENDUE — l'API RDD du TP5
sc = spark.sparkContext
sc.textFile(f"{BASE}/purchases.txt").count()

In [ ]:
# ERREUR ATTENDUE — le cache du TP5 (section 7.5)
ventes.cache()
ventes.count()

In [ ]:
# ERREUR ATTENDUE — le réglage de la jointure du TP7 (section 6)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

Ce réglage-ci, en revanche, est autorisé :

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
print(spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", "auto")

## 6. Delta Lake : une table que l'on peut corriger
### 6.1 Créer la table

In [ ]:
gen.write.mode("overwrite").saveAsTable(f"{CATALOGUE}.{SCHEMA}.ventes")

In [ ]:
%sql
DESCRIBE DETAIL workspace.tp8.ventes

### 6.2 Intégrer des corrections avec MERGE

In [ ]:
table = spark.table(f"{CATALOGUE}.{SCHEMA}.ventes")
# deux ventes saisies avec un montant erroné (remise de 10 % oubliée)…
a_corriger = (table.filter(F.col("id_vente").isin(1, 2))
                   .withColumn("montant", F.round(F.col("montant") * 0.9, 2)))
# … et une vente oubliée
oubliee = spark.createDataFrame(
    [(3_000_001, "2012-12-31", "18:30", "Chicago", "Books", 42.5, "Cash")], table.schema)
corrections = a_corriger.unionByName(oubliee)
corrections.createOrReplaceTempView("corrections")
display(corrections)

In [ ]:
%sql
MERGE INTO workspace.tp8.ventes AS t
USING corrections AS c
ON t.id_vente = c.id_vente
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sql
DESCRIBE HISTORY workspace.tp8.ventes

### 6.3 Remonter le temps

In [ ]:
%sql
-- Remplacez 0 par le numéro de version qui précède le MERGE (colonne version de l'historique)
SELECT id_vente, montant FROM workspace.tp8.ventes VERSION AS OF 0 WHERE id_vente IN (1, 2, 3000001)

In [ ]:
%sql
SELECT id_vente, montant FROM workspace.tp8.ventes WHERE id_vente IN (1, 2, 3000001)

In [ ]:
# À FAIRE — calculez l'écart de CA total entre la version d'avant le MERGE et la version actuelle (une requête SQL ou DataFrame).


## 7. Préparer le tableau de bord
On enregistre le référentiel et un agrégat en tables : le tableau de bord (section 7 de l'énoncé) les interrogera en SQL.

In [ ]:
# À FAIRE — enregistrez regions en table workspace.tp8.regions (saveAsTable), puis créez la table workspace.tp8.ca_region_mois (région, mois, CA) à partir de la table Delta ventes.


## 8. Fin de séance
Le calcul serverless s'arrête seul après une période d'inactivité. Pensez à **mettre en pause** la planification du job (section 8 de l'énoncé) pour ne pas consommer votre quota.